# 02 — Automation: Linux (Ansible) & Windows (PowerShell) + Python

Demonstra a camada de configuração/automação (v0.2):

1. Os módulos `automation/troubleshooting/*.py` chamados **diretamente em
   Python** (sem precisar de um servidor real — health check roda contra um
   `python -m http.server` local descartável, disk check roda contra o
   disco local, service check simula um serviço).
2. A estrutura das **roles Ansible** (`ansible/roles/`) — lida a partir dos
   arquivos, sem executar `ansible-playbook` (que precisa de hosts Linux
   reais / WSL).
3. A estrutura do **módulo PowerShell InfraOps** (`powershell/modules/InfraOps/`)
   — lida a partir dos arquivos, sem precisar rodar Pester/PowerShell real.

In [1]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing PROJECT_LOG.md
    is found. Works whether the notebook is executed from notebooks/ (the
    normal case) or from the repo root."""
    p = start.resolve()
    for _ in range(8):
        if (p / "PROJECT_LOG.md").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not locate project root (PROJECT_LOG.md not found upward from %s)" % start)

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Caterpillar (Terminar)


## Parte 1 — Health check real (Python), contra um servidor HTTP descartável

In [2]:
import subprocess
import sys
import tempfile
import time
from pathlib import Path as _Path

from automation.troubleshooting.health_check import run_health_check, find_free_port, check_port
from automation.troubleshooting.disk_check import check_disk_usage
from automation.troubleshooting.service_check import check_service_status

# Webroot minúsculo e descartável -- serve um único index.html em vez da
# árvore inteira do projeto, para o health check responder instantaneamente.
WEBROOT = _Path(tempfile.mkdtemp(prefix="nb02_webroot_"))
(WEBROOT / "index.html").write_text("<html><body>demo ok</body></html>", encoding="utf-8")

# Servidor HTTP local, só para este notebook ter algo real para checar
# (nenhuma chamada de rede externa, nenhum servidor de produção envolvido).
port = find_free_port()
server_proc = subprocess.Popen(
    [sys.executable, "-m", "http.server", str(port), "--bind", "127.0.0.1"],
    cwd=str(WEBROOT), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Espera o servidor subir (poll simples).
for _ in range(50):
    if check_port(host="127.0.0.1", port=port, timeout=0.2):
        break
    time.sleep(0.1)

result = run_health_check(host="127.0.0.1", port=port, process_name="http.server")
print("health_check.run_health_check() ->")
for k, v in result.items():
    print(f"  {k}: {v}")

health_check.run_health_check() ->
  host: 127.0.0.1
  checked_at: 2026-08-19T23:19:29.321700+00:00
  port: 57556
  port_open: True
  process_name: http.server
  pid: None
  process_running: True
  healthy: True


In [3]:
disk = check_disk_usage(path=str(PROJECT_ROOT.anchor or "/"), threshold_percent=90.0)
print("disk_check.check_disk_usage() ->")
for k, v in disk.items():
    print(f"  {k}: {v}")

disk_check.check_disk_usage() ->
  path: G:\
  checked_at: 2026-08-19T23:19:29.542433+00:00
  total_bytes: 998964215808
  used_bytes: 735397736448
  free_bytes: 263566479360
  percent_used: 73.62
  threshold_percent: 90.0
  alert: False


In [4]:
service = check_service_status(
    service_name="demo-http-server",
    host="127.0.0.1",
    port=port,
    process_name="http.server",
)
print("service_check.check_service_status() ->")
for k, v in service.items():
    print(f"  {k}: {v}")

service_check.check_service_status() ->
  service_name: demo-http-server
  host: 127.0.0.1
  port: 57556
  checked_at: 2026-08-19T23:19:29.663948+00:00
  port_open: True
  process_name: http.server
  process_running: True
  status: up


In [5]:
# Encerra o servidor descartável e limpa o webroot temporário.
import shutil as _shutil2

server_proc.terminate()
try:
    server_proc.wait(timeout=5)
except Exception:
    server_proc.kill()
_shutil2.rmtree(WEBROOT, ignore_errors=True)
print("Servidor de demonstração encerrado.")

Servidor de demonstração encerrado.


## Parte 2 — Estrutura das roles Ansible (leitura estática, sem `ansible-playbook`)

In [6]:
ANSIBLE_DIR = PROJECT_ROOT / "ansible"

if not ANSIBLE_DIR.exists():
    print("ansible/ ainda não existe neste checkout — pulando esta seção.")
else:
    print(f"ansible.cfg presente: {(ANSIBLE_DIR / 'ansible.cfg').exists()}")
    roles_dir = ANSIBLE_DIR / "roles"
    if roles_dir.exists():
        for role_dir in sorted(p for p in roles_dir.iterdir() if p.is_dir()):
            task_files = sorted((role_dir / "tasks").glob("*.yml")) if (role_dir / "tasks").exists() else []
            print(f"\nrole: {role_dir.name}")
            print(f"  tasks/: {[f.name for f in task_files] if task_files else '(vazio)'}")
            for sub in ("handlers", "defaults", "meta"):
                sub_dir = role_dir / sub
                if sub_dir.exists() and any(sub_dir.iterdir()):
                    print(f"  {sub}/: {[f.name for f in sub_dir.iterdir()]}")
    playbooks_dir = ANSIBLE_DIR / "playbooks"
    if playbooks_dir.exists():
        pb_files = sorted(playbooks_dir.glob("*.yml"))
        print(f"\nplaybooks/: {[f.name for f in pb_files] if pb_files else '(vazio ainda — outra trilha em andamento)'}")

ansible.cfg presente: True


role: base
  tasks/: ['firewall.yml', 'hardening.yml', 'main.yml', 'packages.yml', 'users.yml']
  handlers/: ['main.yml']


  defaults/: ['main.yml']
  meta/: ['main.yml']



role: webserver
  tasks/: ['install.yml', 'main.yml']
  handlers/: ['main.yml']


  defaults/: ['main.yml']


  meta/: ['main.yml']

playbooks/: (vazio ainda — outra trilha em andamento)


In [7]:
# Mostra o conteúdo da role "base" (a mais completa) como amostra do padrão
# usado em todas as roles.
base_main = ANSIBLE_DIR / "roles" / "base" / "tasks" / "main.yml"
if base_main.exists():
    print(base_main.read_text(encoding="utf-8"))
else:
    print("role base/tasks/main.yml não encontrada.")

---
# tasks/main.yml — role: base
# Orquestra os subarquivos de tasks. Cada um carrega sua própria tag para
# permitir execução seletiva (ex.: `--tags firewall`).

- name: Assert supported OS family (Debian/Ubuntu)
  ansible.builtin.assert:
    that:
      - ansible_facts['os_family'] == 'Debian'
    fail_msg: >-
      A role base foi escrita e testada (por revisão) para Debian/Ubuntu
      (usa apt + ufw). Adapte tasks/packages.yml e tasks/firewall.yml para
      outras famílias (ex.: RedHat + firewalld) antes de usar em outro SO.
    quiet: true
  tags: [base, always]

- name: Include package installation tasks
  ansible.builtin.import_tasks: packages.yml
  tags: [base, packages]

- name: Include users/groups tasks
  ansible.builtin.import_tasks: users.yml
  tags: [base, users]

- name: Include firewall tasks
  ansible.builtin.import_tasks: firewall.yml
  tags: [base, firewall]
  when: base_firewall_enabled | bool

- name: Include hardening tasks
  ansible.builtin.import_tasks: harde

## Parte 3 — Estrutura do módulo PowerShell InfraOps (leitura estática, sem `pwsh`/Pester)

In [8]:
POWERSHELL_DIR = PROJECT_ROOT / "powershell"

if not POWERSHELL_DIR.exists() or not any(POWERSHELL_DIR.rglob("*")):
    print(
        "powershell/ está vazio neste checkout (trilha 'ansible/powershell' ainda em andamento "
        "em paralelo). Estrutura esperada, por convenção do projeto:\n"
        "  powershell/modules/InfraOps/InfraOps.psd1   (module manifest)\n"
        "  powershell/modules/InfraOps/InfraOps.psm1   (module implementation)\n"
        "  powershell/scripts/*.ps1                    (scripts standalone)\n"
        "  powershell/tests/*.Tests.ps1                (testes Pester)\n"
        "Quando os arquivos existirem, esta célula vai listar seu conteúdo automaticamente."
    )
else:
    for f in sorted(POWERSHELL_DIR.rglob("*")):
        if f.is_file():
            print(f.relative_to(PROJECT_ROOT))
    infraops_dir = POWERSHELL_DIR / "modules" / "InfraOps"
    if infraops_dir.exists():
        for f in sorted(infraops_dir.glob("*.psm1")):
            print(f"\n--- {f.name} (primeiras 40 linhas) ---")
            lines = f.read_text(encoding="utf-8").splitlines()
            print("\n".join(lines[:40]))

## Resumo

- Health/disk/service checks rodaram de verdade em Python, contra um
  processo local descartável — sem precisar de nenhum servidor de produção.
- Ansible e PowerShell foram inspecionados estruturalmente (arquivos,
  tasks, roles) sem executar `ansible-playbook`/`pwsh` — que dependem de
  hosts reais e/ou WSL, fora do escopo deste notebook. A execução real
  acontece via `automation/troubleshooting/remediation.py`
  (`run_ansible_playbook` / `run_powershell_script`), demonstrado no
  Notebook 05 com fallback gracioso quando as ferramentas não estão
  instaladas.